# dARK Core Lib - End-to-End Demo Notebook

This notebook demonstrates the `dark-core-lib` Python library for interacting with the dARK 2.0 blockchain system.

## Prerequisites

1. **Start blockchain stack**: Ensure local chain is running (e.g. dark-env)
2. **Deploy contracts**: Ensure `deployed_contracts.ini` exists in dark-dapp
3. **Configure .env**: Use `.env.integration` or `.env` with contract addresses and admin key
4. **Install package**: `pip install -e .`

## 1. Setup

Set `ENV_FILE_PATH` to the exact env file you want to use (`.env.integration` or `.env`).
This notebook will load only that path and fail fast if the file does not exist.

In [ ]:
import os
from dotenv import load_dotenv

if os.path.exists('../.env.integration'):
    load_dotenv('../.env.integration')
    print('Loaded ../.env.integration')
elif os.path.exists('../.env'):
    load_dotenv('../.env')
    print('Loaded ../.env')
else:
    print('No env file found. Create ../.env.integration or ../.env')


## 2. Initialize Core Client

The `DARKCoreClient` loads configuration from `.env` or `.env.integration` file.

In [ ]:
from dark_core_lib import DARKCoreClient

# Initialize (loads config from .env)
client = DARKCoreClient.from_env(read_only=False)

print(f"\n✅ Connected to blockchain")
print(f"📦 Block number: {client.get_block_number()}")
print(f"💰 Admin balance: {client.get_admin_balance():.4f} ETH")

## 3. Setup Authority with NAANs

The main function `setup_authority()` handles the complete flow:
1. Creates a wallet for the authority if not exists
2. Funds the wallet from admin account
3. Registers the authority on blockchain
4. Authorizes all specified NAANs

In [ ]:
# Generate unique UUID for this demo run (avoids conflicts with previous runs)
UUID = f"demo-authority-{int(time.time())}"
NAANS = ["12345", "67890"]

print(f"Setting up authority: {UUID}")
print(f"NAANs to authorize: {NAANS}")

# Setup authority (creates wallet, registers, authorizes NAANs)
authority = client.setup_authority(UUID, NAANS)

print(f"\n✅ Authority setup complete!")
print(f"📋 UUID: {authority.uuid}")
print(f"🔑 Wallet: {authority.wallet_address}")
print(f"📝 NAANs: {authority.naans}")
print(f"✔️ Active: {authority.active}")

## 4. Query Authority Information

In [ ]:
# Get authority by UUID
auth_info = client.get_authority_by_uuid(UUID)
print(f"Authority by UUID:")
print(f"  Wallet: {auth_info.wallet_address}")
print(f"  NAANs: {auth_info.naans}")
print(f"  Active: {auth_info.active}")

# Get wallet balance
balance = client.get_wallet_balance(UUID)
print(f"  Balance: {balance:.6f} ETH")

In [ ]:
# Check authorization for specific NAAN
for naan in ["12345", "67890", "99999"]:
    is_auth = client.is_authorized_for_naan(UUID, naan)
    status = "✅" if is_auth else "❌"
    print(f"{status} Authorized for NAAN '{naan}': {is_auth}")

## 5. Create ARK

In [ ]:
# ARK parameters - use unique name based on timestamp
NAAN = "12345"
NAME = f"document-{int(time.time())}"
URL = "https://example.com/documents/001"
CID = "QmYwAPJzv5CZsnA625s3Xf2nemtYgPpHdWEz79ojWnPbdG"

print(f"Creating ARK: ark:/{NAAN}/{NAME}")
print(f"  URL: {URL}")
print(f"  CID: {CID}")

# Create ARK
ark = client.create_ark(UUID, NAAN, NAME, URL, CID)

print(f"\n✅ ARK created!")
print(f"  ID: {ark.ark_id}")
print(f"  Owner: {ark.owner}")
print(f"  Created: {ark.created_at}")

## 6. Resolve ARK

In [ ]:
# Resolve ARK to URL
resolved_url = client.resolve_ark(NAAN, NAME)
print(f"🔍 Resolved ark:/{NAAN}/{NAME}")
print(f"   → {resolved_url}")

# Get full ARK info
ark_info = client.get_ark(NAAN, NAME)
print(f"\n📦 Full ARK data:")
print(f"   Name: {ark_info.name}")
print(f"   NAAN: {ark_info.naan}")
print(f"   URL: {ark_info.url}")
print(f"   CID: {ark_info.cid}")
print(f"   Owner: {ark_info.owner}")
print(f"   Created: {ark_info.created_at}")
print(f"   Updated: {ark_info.updated_at}")

## 7. Update ARK

In [ ]:
# Update the ARK with new URL
NEW_URL = "https://example.com/documents/001/v2"
NEW_CID = "QmNewCID123456789"

print(f"Updating ARK: ark:/{NAAN}/{NAME}")
print(f"  New URL: {NEW_URL}")

updated_ark = client.update_ark(UUID, NAAN, NAME, NEW_URL, NEW_CID)

print(f"\n✅ ARK updated!")
print(f"   URL: {updated_ark.url}")
print(f"   Updated: {updated_ark.updated_at}")

## 8. Check ARK Existence

In [ ]:
# Check existence
test_cases = [
    (NAAN, NAME),           # Should exist (our created ARK)
    ("12345", "nonexistent"),   # Should not exist
    ("99999", "test"),          # Should not exist
]

for naan, name in test_cases:
    exists = client.ark_exists(naan, name)
    status = "✅" if exists else "❌"
    print(f"{status} ark:/{naan}/{name} exists: {exists}")

## 9. Read-Only Mode\n
\n
You can also initialize a read-only client (observer-style) with:\n
\n
```python\n
from dark_core_lib import DARKCoreClient\n
ro_client = DARKCoreClient.from_env(read_only=True)\n
```\n

## Summary

The `DARKCoreClient` provides a unified high-level API for:

- **Authority Management**
  - `setup_authority(uuid, naans)` - Complete setup flow
  - `get_authority_by_uuid(uuid)` - Query by UUID
  - `get_authority_by_wallet(address)` - Query by wallet
  - `get_authorized_naans(uuid)` - List NAANs
  - `is_authorized_for_naan(uuid, naan)` - Check authorization

- **ARK Operations**
  - `create_ark(uuid, naan, name, url, cid)` - Create new ARK
  - `update_ark(uuid, naan, name, url, cid)` - Update existing
  - `resolve_ark(naan, name)` - Get URL
  - `get_ark(naan, name)` - Full ARK data
  - `ark_exists(naan, name)` - Check existence

- **Utilities**
  - `get_wallet_balance(uuid)` - Check ETH balance
  - `get_admin_balance()` - Admin account balance
  - `is_connected()` - Blockchain connection status